# TAMP-OS Batch Lithograph Maker v2

Select microscopy images, choose a printer profile, choose Low / Medium / High quality, and generate print-ready files.

**Simple mode:**
- Add images
- Choose a printer profile that represents the machine capacity
- Pick Low, Medium, or High quality
- Choose output folder / format
- Optional: choose a different quality per image for mixed batches
- Generate print files

**Full customization** is still available for print width, nozzle diameter, layer height, relief height, output format, blur, resolution, and base thickness.

> Note: Run in **Jupyter Lab or Jupyter Notebook** - not VS Code notebook viewer (tkinter needs a real display).

### Step 1 - Install dependencies (once)


In [5]:
!pip install numpy pillow scipy numpy-stl

### Step 2  Imports

In [6]:
import os, sys, struct, json, zipfile, math
from pathlib import Path
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter
from stl import mesh
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import threading
print("Imports OK")

Imports OK


### Step 3  Pipeline functions & preset calculator
*(self-contained  no external files needed)*

In [7]:
#  Preset calculator 

def estimate_stl_file_size(resolution, aspect_ratio=1.0):
    """Return an approximate binary STL size in MB for a height-map mesh."""
    cols = max(2, int(resolution))
    rows = max(2, int(round(cols * aspect_ratio)))
    num_tris = (rows - 1) * (cols - 1) * 4 + 2 * ((rows - 1) + (cols - 1)) * 2
    return (84 + 50 * num_tris) / (1024 * 1024)


def format_file_size(size_mb):
    if size_mb < 1:
        return f"{size_mb * 1024:.0f} KB"
    if size_mb < 100:
        return f"{size_mb:.1f} MB"
    return f"{size_mb:.0f} MB"


def calculate_presets(print_size_x, nozzle_mm=0.4, layer_mm=0.12, aspect_ratio=1.0):
    """
    Calculate Low / Medium / High presets from printer physical specs.

    Medium = nozzle-matched (1 px per nozzle width), Low is coarser,
    High oversamples the nozzle to preserve more source texture.
    """
    medium_res = round(print_size_x / nozzle_mm)
    low_res    = round(print_size_x / (nozzle_mm * 2))
    high_res   = round(print_size_x / (nozzle_mm * 0.5))

    def snap(v): return max(64, min(512, int(2 ** round(math.log2(v)))))
    low_res    = snap(low_res)
    medium_res = snap(medium_res)
    high_res   = min(512, snap(high_res))

    def preset(resolution, blur, px_size, feature_text, note):
        size_mb = estimate_stl_file_size(resolution, aspect_ratio)
        return {
            "resolution": resolution,
            "blur": blur,
            "px_size": px_size,
            "file_size_mb": size_mb,
            "file_size": format_file_size(size_mb),
            "description": (f"{resolution} px | ~{format_file_size(size_mb)} STL\n"
                            f"1 px = {px_size:.2f} mm | {feature_text}\n"
                            f"{note}")
        }

    return {
        "Low": preset(low_res, 2.0, print_size_x / low_res,
                      f"~{nozzle_mm*2:.1f} mm features", "Smooth, fastest preview"),
        "Medium": preset(medium_res, 1.2, print_size_x / medium_res,
                         f"~{nozzle_mm:.1f} mm features", "Nozzle-matched starting point"),
        "High": preset(high_res, 0.8, print_size_x / high_res,
                       f"~{nozzle_mm*0.5:.2f} mm features", "Finer texture, larger file"),
    }

# Height map helpers

BEST_PRACTICE = {"base_thickness": 1.0, "relief_height": 3.0}


def smart_size_y(img_path, size_x=100.0):
    try:
        img = Image.open(img_path); w, h = img.size
        return round(size_x * h / w, 1)
    except Exception: return 75.0


def analyze_image_quality(img_path):
    """Measure source image size, exposure spread, and edge detail for preset guidance."""
    img = Image.open(img_path).convert("L")
    w, h = img.size
    analysis_img = img.copy()
    analysis_img.thumbnail((1024, 1024), Image.LANCZOS)
    arr = np.array(analysis_img, dtype=np.float32)
    p1, p5, p95, p99 = np.percentile(arr, [1, 5, 95, 99])
    dynamic_range = float(p95 - p5)
    clipped_dark = float(np.mean(arr <= 3) * 100)
    clipped_light = float(np.mean(arr >= 252) * 100)
    gy, gx = np.gradient(arr / 255.0)
    edge_detail = float(np.percentile(np.hypot(gx, gy), 90))

    if dynamic_range < 45:
        exposure = "low contrast"
    elif clipped_dark + clipped_light > 8:
        exposure = "clipped highlights/shadows"
    elif dynamic_range > 170:
        exposure = "high dynamic range"
    else:
        exposure = "balanced contrast"

    if edge_detail > 0.075:
        detail = "fine edge detail/noise"
    elif edge_detail < 0.025:
        detail = "smooth gradients"
    else:
        detail = "moderate detail"

    if min(w, h) < 512:
        pixel_note = "small source image"
    elif min(w, h) > 1800:
        pixel_note = "large source image"
    else:
        pixel_note = "moderate source image"

    return {
        "width": w, "height": h, "megapixels": (w * h) / 1_000_000,
        "dynamic_range": dynamic_range, "exposure": exposure,
        "edge_detail": edge_detail, "detail": detail,
        "clipped_total": clipped_dark + clipped_light, "pixel_note": pixel_note,
    }


def recommend_preset_from_analysis(analysis):
    if analysis["pixel_note"] == "small source image" or analysis["detail"] == "smooth gradients":
        preset = "Low"
    elif analysis["detail"] == "fine edge detail/noise" or analysis["pixel_note"] == "large source image":
        preset = "High"
    else:
        preset = "Medium"

    blur_adjust = ""
    if analysis["detail"] == "fine edge detail/noise":
        blur_adjust = "use a little more blur if the surface looks noisy"
    elif analysis["exposure"] == "low contrast":
        blur_adjust = "consider slightly less blur after contrast stretch"
    elif analysis["exposure"] == "clipped highlights/shadows":
        blur_adjust = "check relief; clipped tones can become flat plateaus"
    return preset, blur_adjust


def summarize_image_recommendations(image_paths):
    if not image_paths:
        return "Add images to see source pixel count and dynamic-range guidance."
    analyses = []
    for path in image_paths[:5]:
        try:
            analyses.append(analyze_image_quality(path))
        except Exception:
            pass
    if not analyses:
        return "Could not read image quality yet. Check that the selected files are valid images."

    presets = [recommend_preset_from_analysis(a)[0] for a in analyses]
    recommended = max(["Low", "Medium", "High"], key=presets.count)
    avg_mp = sum(a["megapixels"] for a in analyses) / len(analyses)
    avg_range = sum(a["dynamic_range"] for a in analyses) / len(analyses)
    traits = sorted(set(a["exposure"] for a in analyses) | set(a["detail"] for a in analyses))
    _, blur_note = recommend_preset_from_analysis(analyses[0])
    note = f"Image check: ~{avg_mp:.1f} MP, dynamic range {avg_range:.0f}/255, {', '.join(traits[:3])}. Try {recommended}."
    if blur_note:
        note += f" {blur_note}."
    if len(image_paths) > len(analyses):
        note += f" Based on first {len(analyses)} images."
    return note
def image_to_heightmap(image_path, output_size=(512,512), invert=False,
                       blur_sigma=1.0, contrast_percentile=(2.0,98.0),
                       preserve_aspect=True, flip=True):
    img = Image.open(image_path).convert("L")
    orig_w, orig_h = img.size
    if preserve_aspect and orig_w != orig_h:
        target_w = output_size[0]
        target_h = round(target_w * orig_h / orig_w)
        actual_size = (target_w, target_h)
        print(f"  [i] {orig_w}x{orig_h} -> {target_w}x{target_h}")
    else:
        actual_size = output_size
    img = img.resize(actual_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)
    lo = np.percentile(arr, contrast_percentile[0])
    hi = np.percentile(arr, contrast_percentile[1])
    arr = np.clip((arr - lo) / (hi - lo + 1e-8), 0.0, 1.0)
    if blur_sigma > 0:
        arr = gaussian_filter(arr, sigma=blur_sigma)
    if invert:
        arr = 1.0 - arr
    if flip:
        arr = np.flipud(arr)
    return arr

def heightmap_to_stl(heightmap, output_path, base_thickness_mm=1.0,
                     max_relief_mm=3.0, physical_size_mm=(100.0,100.0)):
    rows, cols = heightmap.shape
    dx = physical_size_mm[0] / (cols - 1)
    dy = physical_size_mm[1] / (rows - 1)
    hm_ratio = cols / rows
    print_ratio = physical_size_mm[0] / physical_size_mm[1]
    if abs(hm_ratio - print_ratio) > 0.05:
        print(f"  [WARNING] Aspect mismatch - consider size_y={round(physical_size_mm[0]/hm_ratio,1)}")
    z_top = base_thickness_mm + heightmap * max_relief_mm
    num_tris = (rows-1)*(cols-1)*4 + 2*((rows-1)+(cols-1))*2
    litho_mesh = mesh.Mesh(np.zeros(num_tris, dtype=mesh.Mesh.dtype))
    tri_idx = 0
    def add_tri(v0, v1, v2):
        nonlocal tri_idx
        litho_mesh.vectors[tri_idx] = [v0, v1, v2]
        tri_idx += 1
    for r in range(rows-1):
        for c in range(cols-1):
            x0,y0 = c*dx, r*dy
            x1 = (c+1)*dx
            x2,y2 = c*dx, (r+1)*dy
            x3,y3 = (c+1)*dx, (r+1)*dy
            z00,z10,z01,z11 = z_top[r,c],z_top[r,c+1],z_top[r+1,c],z_top[r+1,c+1]
            add_tri([x0,y0,z00],[x1,y0,z10],[x2,y2,z01])
            add_tri([x1,y0,z10],[x3,y3,z11],[x2,y2,z01])
            add_tri([x0,y0,0],[x2,y2,0],[x1,y0,0])
            add_tri([x1,y0,0],[x2,y2,0],[x3,y3,0])
    xmax,ymax = (cols-1)*dx,(rows-1)*dy
    for c in range(cols-1):
        x0,x1 = c*dx,(c+1)*dx
        z0,z1 = z_top[0,c],z_top[0,c+1]
        add_tri([x0,0,0],[x1,0,0],[x0,0,z0])
        add_tri([x1,0,0],[x1,0,z1],[x0,0,z0])
    for c in range(cols-1):
        x0,x1 = c*dx,(c+1)*dx
        z0,z1 = z_top[-1,c],z_top[-1,c+1]
        add_tri([x0,ymax,0],[x0,ymax,z0],[x1,ymax,0])
        add_tri([x1,ymax,0],[x0,ymax,z0],[x1,ymax,z1])
    for r in range(rows-1):
        y0,y1 = r*dy,(r+1)*dy
        z0,z1 = z_top[r,0],z_top[r+1,0]
        add_tri([0,y0,0],[0,y0,z0],[0,y1,0])
        add_tri([0,y1,0],[0,y0,z0],[0,y1,z1])
    for r in range(rows-1):
        y0,y1 = r*dy,(r+1)*dy
        z0,z1 = z_top[r,-1],z_top[r+1,-1]
        add_tri([xmax,y0,0],[xmax,y1,0],[xmax,y0,z0])
        add_tri([xmax,y1,0],[xmax,y1,z1],[xmax,y0,z0])
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    litho_mesh.save(str(output_path))
    print(f"  [OK] STL -> {output_path}")
    return Path(output_path)

#  Format converters 

def stl_to_3mf(stl_path):
    from stl import mesh as stl_mesh
    m = stl_mesh.Mesh.from_file(str(stl_path))
    vertices = []; vert_map = {}; indices = []
    for tri in m.vectors:
        face = []
        for v in tri:
            key = tuple(float(x) for x in v)
            if key not in vert_map:
                vert_map[key] = len(vertices)
                vertices.append(key)
            face.append(vert_map[key])
        indices.append(face)
    vx = " ".join(
        f'<vertex x="{v[0]:.4f}" y="{v[1]:.4f}" z="{v[2]:.4f}"/>' for v in vertices)
    tx = " ".join(
        f'<triangle v1="{t[0]}" v2="{t[1]}" v3="{t[2]}"/>' for t in indices)
    ns = "http://schemas.microsoft.com/3dmanufacturing/core/2015/02"
    xml = (f'<+xml version="1.0" encoding="UTF-8"+>'
           f'<model unit="millimeter" xmlns="{ns}">'
           f'<resources><object id="1" type="model"><mesh>'
           f'<vertices>{vx}</vertices><triangles>{tx}</triangles>'
           f'</mesh></object></resources><build><item objectid="1"/></build></model>')
    out = Path(str(stl_path).replace(".stl", ".3mf"))
    ct_ns = "http://schemas.openxmlformats.org/package/2006/content-types"
    rel_ns = "http://schemas.openxmlformats.org/package/2006/relationships"
    rel_type = "http://schemas.microsoft.com/3dmanufacturing/2013/01/3dmodel"
    with zipfile.ZipFile(str(out), "w", zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("3D/3dmodel.model", xml)
        zf.writestr("[Content_Types].xml",
            f'<+xml version="1.0"+><Types xmlns="{ct_ns}">'
            f'<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>'
            f'<Default Extension="model" ContentType="application/vnd.ms-package.3dmanufacturing-3dmodel+xml"/>'
            f'</Types>')
        zf.writestr("_rels/.rels",
            f'<+xml version="1.0"+><Relationships xmlns="{rel_ns}">'
            f'<Relationship Type="{rel_type}" Target="/3D/3dmodel.model" Id="r1"/>'
            f'</Relationships>')
    print(f"  [OK] 3MF -> {out}")
    return out

def stl_to_glb(stl_path):
    from stl import mesh as stl_mesh
    m = stl_mesh.Mesh.from_file(str(stl_path))
    verts = m.vectors.reshape(-1, 3).astype(np.float32)
    indices = np.arange(len(verts), dtype=np.uint32)
    def pad4(b): return b + b'\x00' * ((4 - len(b) % 4) % 4)
    ib = pad4(indices.tobytes())
    vb = pad4(verts.tobytes())
    bd = ib + vb
    gltf = {
        "asset": {"version": "2.0", "generator": "TAMP-OS"}, "scene": 0,
        "scenes": [{"nodes": [0]}], "nodes": [{"mesh": 0}],
        "meshes": [{"primitives": [{"attributes": {"POSITION": 1}, "indices": 0}]}],
        "accessors": [
            {"bufferView": 0, "componentType": 5125, "count": len(indices), "type": "SCALAR"},
            {"bufferView": 1, "componentType": 5126, "count": len(verts), "type": "VEC3",
             "min": verts.min(0).tolist(), "max": verts.max(0).tolist()}],
        "bufferViews": [
            {"buffer": 0, "byteOffset": 0, "byteLength": len(ib), "target": 34963},
            {"buffer": 0, "byteOffset": len(ib), "byteLength": len(vb), "target": 34962}],
        "buffers": [{"byteLength": len(bd)}]
    }
    jb = pad4(json.dumps(gltf, separators=(",", ":")).encode())
    def chunk(d, t): return struct.pack("<II", len(d), t) + d
    out = Path(str(stl_path).replace(".stl", ".glb"))
    with open(str(out), "wb") as f:
        f.write(struct.pack("<III", 0x46546C67, 2, 12 + 8 + len(jb) + 8 + len(bd))
                + chunk(jb, 0x4E4F534A) + chunk(bd, 0x004E4942))
    print(f"  [OK] GLB -> {out}")
    return out

def convert_stl(stl_path, formats):
    if formats.get("3mf"): stl_to_3mf(stl_path)
    if formats.get("glb"):  stl_to_glb(stl_path)
    if not formats.get("stl"): stl_path.unlink(missing_ok=True)

print("All pipeline functions loaded OK.")

All pipeline functions loaded OK.


## Step 4  Launch the GUI
Run this cell to open the window.

In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import threading
import math

#  Preset calculator 

def estimate_stl_file_size(resolution, aspect_ratio=1.0):
    """Return an approximate binary STL size in MB for a height-map mesh."""
    cols = max(2, int(resolution))
    rows = max(2, int(round(cols * aspect_ratio)))
    num_tris = (rows - 1) * (cols - 1) * 4 + 2 * ((rows - 1) + (cols - 1)) * 2
    return (84 + 50 * num_tris) / (1024 * 1024)


def format_file_size(size_mb):
    if size_mb < 1:
        return f"{size_mb * 1024:.0f} KB"
    if size_mb < 100:
        return f"{size_mb:.1f} MB"
    return f"{size_mb:.0f} MB"


def calculate_presets(print_size_x, nozzle_mm=0.4, layer_mm=0.12, aspect_ratio=1.0):
    """
    Calculate Low / Medium / High presets from printer physical specs.

    Medium = nozzle-matched (1 px per nozzle width), Low is coarser,
    High oversamples the nozzle to preserve more source texture.
    """
    medium_res = round(print_size_x / nozzle_mm)
    low_res    = round(print_size_x / (nozzle_mm * 2))
    high_res   = round(print_size_x / (nozzle_mm * 0.5))

    def snap(v): return max(64, min(512, int(2 ** round(math.log2(v)))))
    low_res    = snap(low_res)
    medium_res = snap(medium_res)
    high_res   = min(512, snap(high_res))

    def preset(resolution, blur, px_size, feature_text, note):
        size_mb = estimate_stl_file_size(resolution, aspect_ratio)
        return {
            "resolution": resolution,
            "blur": blur,
            "px_size": px_size,
            "file_size_mb": size_mb,
            "file_size": format_file_size(size_mb),
            "description": (f"{note}\n"
                            f"~{format_file_size(size_mb)} STL | {resolution} px")
        }

    return {
        "Low": preset(low_res, 2.0, print_size_x / low_res,
                      f"~{nozzle_mm*2:.1f} mm features", "Quick draft"),
        "Medium": preset(medium_res, 1.2, print_size_x / medium_res,
                         f"~{nozzle_mm:.1f} mm features", "Recommended"),
        "High": preset(high_res, 0.8, print_size_x / high_res,
                       f"~{nozzle_mm*0.5:.2f} mm features", "Maximum detail"),
    }

BEST_PRACTICE = {"base_thickness": 1.0, "relief_height": 3.0}


def smart_size_y(img_path, size_x=100.0):
    try:
        img = Image.open(img_path); w, h = img.size
        return round(size_x * h / w, 1)
    except Exception: return 75.0


def analyze_image_quality(img_path):
    """Measure source image size, exposure spread, and edge detail for preset guidance."""
    img = Image.open(img_path).convert("L")
    w, h = img.size
    analysis_img = img.copy()
    analysis_img.thumbnail((1024, 1024), Image.LANCZOS)
    arr = np.array(analysis_img, dtype=np.float32)
    p1, p5, p95, p99 = np.percentile(arr, [1, 5, 95, 99])
    dynamic_range = float(p95 - p5)
    clipped_dark = float(np.mean(arr <= 3) * 100)
    clipped_light = float(np.mean(arr >= 252) * 100)
    gy, gx = np.gradient(arr / 255.0)
    edge_detail = float(np.percentile(np.hypot(gx, gy), 90))

    if dynamic_range < 45:
        exposure = "low contrast"
    elif clipped_dark + clipped_light > 8:
        exposure = "clipped highlights/shadows"
    elif dynamic_range > 170:
        exposure = "high dynamic range"
    else:
        exposure = "balanced contrast"

    if edge_detail > 0.075:
        detail = "fine edge detail/noise"
    elif edge_detail < 0.025:
        detail = "smooth gradients"
    else:
        detail = "moderate detail"

    if min(w, h) < 512:
        pixel_note = "small source image"
    elif min(w, h) > 1800:
        pixel_note = "large source image"
    else:
        pixel_note = "moderate source image"

    return {
        "width": w, "height": h, "megapixels": (w * h) / 1_000_000,
        "dynamic_range": dynamic_range, "exposure": exposure,
        "edge_detail": edge_detail, "detail": detail,
        "clipped_total": clipped_dark + clipped_light, "pixel_note": pixel_note,
    }


def recommend_preset_from_analysis(analysis):
    if analysis["pixel_note"] == "small source image" or analysis["detail"] == "smooth gradients":
        preset = "Low"
    elif analysis["detail"] == "fine edge detail/noise" or analysis["pixel_note"] == "large source image":
        preset = "High"
    else:
        preset = "Medium"

    blur_adjust = ""
    if analysis["detail"] == "fine edge detail/noise":
        blur_adjust = "use a little more blur if the surface looks noisy"
    elif analysis["exposure"] == "low contrast":
        blur_adjust = "consider slightly less blur after contrast stretch"
    elif analysis["exposure"] == "clipped highlights/shadows":
        blur_adjust = "check relief; clipped tones can become flat plateaus"
    return preset, blur_adjust


def summarize_image_recommendations(image_paths):
    if not image_paths:
        return "Add images to get a quality suggestion."
    analyses = []
    for path in image_paths:
        try:
            analyses.append(analyze_image_quality(path))
        except Exception:
            pass
    if not analyses:
        return "Could not read image quality yet. Check that the selected files are valid images."

    presets = [recommend_preset_from_analysis(a)[0] for a in analyses]
    counts = {name: presets.count(name) for name in ["Low", "Medium", "High"]}
    recommended = max(["Low", "Medium", "High"], key=lambda name: (counts[name], -["Low", "Medium", "High"].index(name)))
    avg_mp = sum(a["megapixels"] for a in analyses) / len(analyses)
    avg_range = sum(a["dynamic_range"] for a in analyses) / len(analyses)
    traits = sorted(set(a["exposure"] for a in analyses) | set(a["detail"] for a in analyses))
    count_text = ", ".join(f"{counts[name]} {name}" for name in ["Low", "Medium", "High"] if counts[name])
    note = f"Suggested batch quality: {recommended}. {count_text} across {len(analyses)} image(s); average contrast range {avg_range:.0f}/255."
    if len(set(presets)) > 1:
        note += " Images vary; Medium is usually the safest batch choice."
    else:
        _, blur_note = recommend_preset_from_analysis(analyses[0])
        if blur_note:
            note += f" {blur_note}."
    return note

#  Scrollable base window 

class ScrollableFrame(tk.Frame):
    """A frame that becomes scrollable when content exceeds screen height."""
    def __init__(self, parent):
        super().__init__(parent)
        self.canvas = tk.Canvas(self, borderwidth=0, highlightthickness=0)
        self.scrollbar = tk.Scrollbar(self, orient="vertical",
                                      command=self.canvas.yview)
        self.inner = tk.Frame(self.canvas)

        self.canvas.configure(yscrollcommand=self.scrollbar.set)
        self.scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        self.window_id = self.canvas.create_window(
            (0, 0), window=self.inner, anchor="nw")

        self.inner.bind("<Configure>", self._on_frame_configure)
        self.canvas.bind("<Configure>", self._on_canvas_configure)

        # Mouse wheel scrolling
        self.canvas.bind_all("<MouseWheel>",
            lambda e: self.canvas.yview_scroll(-1*(e.delta//120), "units"))
        self.canvas.bind_all("<Button-4>",
            lambda e: self.canvas.yview_scroll(-1, "units"))
        self.canvas.bind_all("<Button-5>",
            lambda e: self.canvas.yview_scroll(1, "units"))

    def _on_frame_configure(self, event):
        self.canvas.configure(scrollregion=self.canvas.bbox("all"))

    def _on_canvas_configure(self, event):
        self.canvas.itemconfig(self.window_id, width=event.width)


#  Main App 

class TAMPBatchGUIv2:
    def __init__(self, root):
        self.root = root
        self.root.title("TAMP-OS Batch Lithograph Maker v2")

        # Set window size to 90% of screen height
        sw = root.winfo_screenwidth()
        sh = root.winfo_screenheight()
        win_w = 660
        win_h = min(900, int(sh * 0.88))
        root.geometry(f"{win_w}x{win_h}")
        root.resizable(False, True)

        # Scrollable container
        self.sf = ScrollableFrame(root)
        self.sf.pack(fill=tk.BOTH, expand=True)
        self.f = self.sf.inner   # all widgets go here

        self.image_paths = []
        self.running = False
        self.advanced_open = tk.BooleanVar(value=False)
        self.selected_preset = tk.StringVar(value="Medium")
        self._build_ui()
        self._update_preset_display()

    def _build_ui(self):
        f = self.f
        pad = dict(padx=14, pady=5)
        row = 0

        tk.Label(f, text="TAMP-OS Lithograph Maker",
                 font=("Helvetica", 15, "bold")).grid(
            row=row, column=0, columnspan=3, pady=(16,2))
        row += 1
        tk.Label(f, text="Upload images, choose quality, generate print files.",
                 font=("Helvetica", 9), fg="gray").grid(
            row=row, column=0, columnspan=3, pady=(0,10))
        row += 1

        # 1. Images
        tk.Label(f, text="1. Images", font=("Helvetica", 11, "bold")).grid(
            row=row, column=0, columnspan=3, sticky="w", **pad)
        row += 1
        fl = tk.Frame(f)
        fl.grid(row=row, column=0, columnspan=3, padx=14, pady=(0,4), sticky="ew")
        sb = tk.Scrollbar(fl, orient=tk.VERTICAL)
        self.listbox = tk.Listbox(fl, width=62, height=5,
                                  yscrollcommand=sb.set, selectmode=tk.EXTENDED)
        sb.config(command=self.listbox.yview)
        self.listbox.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        sb.pack(side=tk.RIGHT, fill=tk.Y)
        row += 1
        bf = tk.Frame(f)
        bf.grid(row=row, column=0, columnspan=3, sticky="w", padx=14, pady=(0,8))
        tk.Button(bf, text="Add images", command=self._add_images,
                  width=16).pack(side=tk.LEFT, padx=(0,6))
        tk.Button(bf, text="Remove selected", command=self._remove_selected,
                  width=16).pack(side=tk.LEFT)
        row += 1

        # 2. Printer profile
        ttk.Separator(f, orient="horizontal").grid(
            row=row, column=0, columnspan=3, sticky="ew", padx=10, pady=6)
        row += 1
        tk.Label(f, text="2. Printer", font=("Helvetica", 11, "bold")).grid(
            row=row, column=0, columnspan=3, sticky="w", **pad)
        row += 1
        profile_row = tk.Frame(f)
        profile_row.grid(row=row, column=0, columnspan=3, sticky="w", padx=14, pady=(0,4))
        self.profile_var = tk.StringVar(value="Standard FDM / 0.4 mm nozzle")
        profiles = [
            "Standard FDM / 0.4 mm nozzle",
            "Fine-detail FDM / 0.2 mm nozzle",
            "Large-nozzle FDM / 0.6 mm nozzle",
            "Custom",
        ]
        ttk.OptionMenu(profile_row, self.profile_var, self.profile_var.get(),
                       *profiles, command=self._apply_machine_profile).pack(side=tk.LEFT)
        self.machine_summary_var = tk.StringVar(value="")
        tk.Label(profile_row, textvariable=self.machine_summary_var,
                 fg="gray", font=("Helvetica", 8)).pack(side=tk.LEFT, padx=(10,0))
        row += 1

        # Defaults controlled by the profile; editable in Full customization.
        self.size_x_var = tk.StringVar(value="100")
        self.nozzle_var = tk.StringVar(value="0.4")
        self.layer_var = tk.StringVar(value="0.12")
        self.size_y_var = tk.StringVar(value="auto")
        self.relief_var = tk.StringVar(value="3.0")
        self.flip_var = tk.BooleanVar(value=True)
        self.invert_var = tk.BooleanVar(value=False)
        self.output_var = tk.StringVar(value=str(Path.home() / "TAMP_output"))
        self.fmt_stl = tk.BooleanVar(value=True)
        self.fmt_3mf = tk.BooleanVar(value=False)
        self.fmt_glb = tk.BooleanVar(value=False)

        # 3. Quality presets
        ttk.Separator(f, orient="horizontal").grid(
            row=row, column=0, columnspan=3, sticky="ew", padx=10, pady=6)
        row += 1
        tk.Label(f, text="3. Quality", font=("Helvetica", 11, "bold")).grid(
            row=row, column=0, columnspan=3, sticky="w", **pad)
        row += 1
        preset_frame = tk.Frame(f)
        preset_frame.grid(row=row, column=0, columnspan=3, padx=14, pady=(0,4), sticky="ew")
        self.preset_descriptions = {}
        colors = {"Low": "#edf6ed", "Medium": "#edf2ff", "High": "#fff3eb"}
        for col, preset in enumerate(["Low", "Medium", "High"]):
            pf = tk.Frame(preset_frame, relief="groove", bd=1, bg=colors[preset])
            pf.grid(row=0, column=col, padx=5, pady=4, sticky="nsew")
            preset_frame.columnconfigure(col, weight=1)
            tk.Radiobutton(pf, text=preset, variable=self.selected_preset,
                           value=preset, font=("Helvetica", 12, "bold"),
                           bg=colors[preset], command=self._update_preset_display).pack(
                fill=tk.X, padx=4, pady=(8,2))
            desc_var = tk.StringVar(value="")
            self.preset_descriptions[preset] = desc_var
            tk.Label(pf, textvariable=desc_var, fg="#444444",
                     font=("Helvetica", 8), justify="center",
                     wraplength=160, bg=colors[preset]).pack(
                padx=8, pady=(0,8), fill=tk.X)
        row += 1
        self.recommendation_var = tk.StringVar(value="Add images to get a quality suggestion.")
        tk.Label(f, textvariable=self.recommendation_var,
                 fg="#2a5d9f", font=("Helvetica", 9), justify="left",
                 wraplength=610).grid(row=row, column=0, columnspan=3,
                                      sticky="w", padx=14, pady=(2,4))
        row += 1
        self.per_image_quality_open = tk.BooleanVar(value=False)
        self.per_image_quality_vars = {}
        self.per_image_suggestion_vars = {}
        tk.Checkbutton(f, text="Choose different quality for different images",
                       variable=self.per_image_quality_open,
                       command=self._toggle_per_image_quality).grid(
            row=row, column=0, columnspan=3, sticky="w", padx=14, pady=(0,2))
        row += 1
        self.per_image_quality_frame = tk.Frame(f, relief="groove", bd=1)
        self.per_image_quality_row = row
        row += 1

        # 4. Output files
        ttk.Separator(f, orient="horizontal").grid(
            row=row, column=0, columnspan=3, sticky="ew", padx=10, pady=6)
        row += 1
        tk.Label(f, text="4. Output files", font=("Helvetica", 11, "bold")).grid(
            row=row, column=0, columnspan=3, sticky="w", **pad)
        row += 1
        output_frame = tk.Frame(f)
        output_frame.grid(row=row, column=0, columnspan=3, sticky="ew", padx=14, pady=(0,4))
        tk.Label(output_frame, text="Output folder:").grid(row=0, column=0, sticky="w", pady=2)
        tk.Entry(output_frame, textvariable=self.output_var, width=42).grid(
            row=0, column=1, sticky="w", padx=6, pady=2)
        tk.Button(output_frame, text="Browse", command=self._browse_output).grid(
            row=0, column=2, sticky="w", padx=(4,0), pady=2)
        tk.Label(output_frame, text="Output format:").grid(row=1, column=0, sticky="w", pady=2)
        fmt_frame = tk.Frame(output_frame)
        fmt_frame.grid(row=1, column=1, columnspan=2, sticky="w", padx=6, pady=2)
        tk.Checkbutton(fmt_frame, text=".STL", variable=self.fmt_stl).pack(side=tk.LEFT, padx=(0,12))
        tk.Checkbutton(fmt_frame, text=".3MF", variable=self.fmt_3mf).pack(side=tk.LEFT, padx=(0,12))
        tk.Checkbutton(fmt_frame, text=".GLB", variable=self.fmt_glb).pack(side=tk.LEFT, padx=(0,12))
        row += 1

        # Full customization keeps every knob out of the happy path.
        ttk.Separator(f, orient="horizontal").grid(
            row=row, column=0, columnspan=3, sticky="ew", padx=10, pady=6)
        row += 1
        adv_hdr = tk.Frame(f)
        adv_hdr.grid(row=row, column=0, columnspan=3, padx=14, pady=(0,2), sticky="w")
        tk.Checkbutton(adv_hdr, text="Full customization",
                       variable=self.advanced_open,
                       font=("Helvetica", 10, "bold"),
                       command=self._toggle_advanced).pack(side=tk.LEFT)
        tk.Label(adv_hdr, text="optional printer, file, and mesh settings",
                 fg="#666666", font=("Helvetica", 8)).pack(side=tk.LEFT, padx=6)
        row += 1

        self.adv_panel = tk.LabelFrame(f, text="Full configuration",
                                       font=("Helvetica", 9), padx=8, pady=6)
        adv_params = [
            ("Print width (mm)",     "size_x",        self.size_x_var, "Physical width of the lithograph"),
            ("Print height (mm)",    "size_y",        self.size_y_var, "auto keeps the image aspect ratio"),
            ("Nozzle diameter (mm)", "nozzle",        self.nozzle_var, "Defines printable XY detail"),
            ("Layer height (mm)",    "layer",         self.layer_var, "Defines relief height steps"),
            ("Relief height (mm)",   "relief",        self.relief_var, "Height from darkest to lightest tone"),
            ("Resolution (px)",      "resolution",    tk.StringVar(value="256"), "Overrides selected quality"),
            ("Blur (sigma)",         "blur",          tk.StringVar(value="1.2"), "Smooths image noise before meshing"),
            ("Base thickness (mm)",  "base_thickness",tk.StringVar(value="1.0"), "Solid base below relief"),
        ]
        self.adv_vars = {}
        for i, (label, key, var, hint) in enumerate(adv_params):
            self.adv_vars[key] = var
            tk.Label(self.adv_panel, text=label+":").grid(row=i, column=0, sticky="w", pady=2)
            entry = tk.Entry(self.adv_panel, textvariable=var, width=12)
            entry.grid(row=i, column=1, sticky="w", padx=6, pady=2)
            if key in ("size_x", "nozzle", "layer"):
                entry.bind("<FocusOut>", lambda e: self._mark_custom_profile())
            elif key in ("size_y", "relief"):
                entry.bind("<FocusOut>", lambda e: self._update_preset_display())
            tk.Label(self.adv_panel, text=hint, fg="gray", font=("Helvetica", 8)).grid(
                row=i, column=2, sticky="w", padx=(0,8), pady=2)

        orient_row = len(adv_params)
        tk.Checkbutton(self.adv_panel,
                       text="Flip vertically (recommended - matches image orientation)",
                       variable=self.flip_var).grid(
            row=orient_row, column=0, columnspan=4, sticky="w", pady=(8,2))
        tk.Checkbutton(self.adv_panel,
                       text="Invert relief (dark areas raised - for some TEM images)",
                       variable=self.invert_var).grid(
            row=orient_row + 1, column=0, columnspan=4, sticky="w", pady=2)
        self.adv_panel_row = row
        row += 1

        # Generate
        ttk.Separator(f, orient="horizontal").grid(
            row=row, column=0, columnspan=3, sticky="ew", padx=10, pady=6)
        row += 1
        self.progress = ttk.Progressbar(f, orient=tk.HORIZONTAL, length=430, mode="determinate")
        self.progress.grid(row=row, column=0, columnspan=3, padx=14, pady=4)
        row += 1
        self.status_var = tk.StringVar(value="Ready.")
        tk.Label(f, textvariable=self.status_var, fg="gray", font=("Helvetica", 9)).grid(
            row=row, column=0, columnspan=3, pady=(0,4))
        row += 1
        self.run_btn = tk.Button(f, text="Generate print files",
                                 font=("Helvetica", 12, "bold"),
                                 bg="#2a7ae2", fg="white",
                                 activebackground="#1a5fbf", activeforeground="white",
                                 command=self._run, width=24, height=2)
        self.run_btn.grid(row=row, column=0, columnspan=3, pady=(4,16))
        self._apply_machine_profile(self.profile_var.get())

    #  Preset logic 

    def _suggest_quality_for_image(self, img_path):
        try:
            analysis = analyze_image_quality(img_path)
            return recommend_preset_from_analysis(analysis)[0]
        except Exception:
            return self.selected_preset.get()

    def _toggle_per_image_quality(self):
        if self.per_image_quality_open.get():
            self.per_image_quality_frame.grid(row=self.per_image_quality_row, column=0, columnspan=3,
                                              sticky="ew", padx=14, pady=(0,8))
        else:
            self.per_image_quality_frame.grid_remove()
        self._refresh_per_image_quality_table()

    def _refresh_per_image_quality_table(self):
        for child in self.per_image_quality_frame.winfo_children():
            child.destroy()
        current_paths = set(self.image_paths)
        for path in list(self.per_image_quality_vars):
            if path not in current_paths:
                self.per_image_quality_vars.pop(path, None)
                self.per_image_suggestion_vars.pop(path, None)
        if not self.image_paths:
            tk.Label(self.per_image_quality_frame, text="Add images to choose quality per image.",
                     fg="gray", font=("Helvetica", 8)).grid(row=0, column=0, sticky="w", padx=8, pady=6)
            return
        tk.Label(self.per_image_quality_frame, text="Image", font=("Helvetica", 8, "bold")).grid(
            row=0, column=0, sticky="w", padx=8, pady=(6,2))
        tk.Label(self.per_image_quality_frame, text="Suggested", font=("Helvetica", 8, "bold")).grid(
            row=0, column=1, sticky="w", padx=8, pady=(6,2))
        tk.Label(self.per_image_quality_frame, text="Use", font=("Helvetica", 8, "bold")).grid(
            row=0, column=2, sticky="w", padx=8, pady=(6,2))
        for i, path in enumerate(self.image_paths, start=1):
            suggested = self._suggest_quality_for_image(path)
            if path not in self.per_image_quality_vars:
                self.per_image_quality_vars[path] = tk.StringVar(value=suggested)
            self.per_image_suggestion_vars[path] = tk.StringVar(value=suggested)
            name = Path(path).name
            if len(name) > 34:
                name = name[:31] + "..."
            tk.Label(self.per_image_quality_frame, text=name, anchor="w").grid(
                row=i, column=0, sticky="w", padx=8, pady=2)
            tk.Label(self.per_image_quality_frame, textvariable=self.per_image_suggestion_vars[path],
                     fg="gray").grid(row=i, column=1, sticky="w", padx=8, pady=2)
            ttk.OptionMenu(self.per_image_quality_frame, self.per_image_quality_vars[path],
                           self.per_image_quality_vars[path].get(), "Low", "Medium", "High").grid(
                row=i, column=2, sticky="w", padx=8, pady=2)

    def _apply_machine_profile(self, profile):
        profiles = {
            "Standard FDM / 0.4 mm nozzle": ("0.4", "0.12", "100"),
            "Fine-detail FDM / 0.2 mm nozzle": ("0.2", "0.08", "80"),
            "Large-nozzle FDM / 0.6 mm nozzle": ("0.6", "0.20", "120"),
        }
        if profile in profiles:
            nozzle, layer, width = profiles[profile]
            self.nozzle_var.set(nozzle)
            self.layer_var.set(layer)
            self.size_x_var.set(width)
            self.machine_summary_var.set(f"{nozzle} mm nozzle, {layer} mm layers, {width} mm wide")
        else:
            self.machine_summary_var.set("using values in Full customization")
        self._update_preset_display()

    def _mark_custom_profile(self):
        if hasattr(self, "profile_var"):
            self.profile_var.set("Custom")
        if hasattr(self, "machine_summary_var"):
            self.machine_summary_var.set("using values in Full customization")
        self._update_preset_display()

    def _update_preset_display(self):
        try: size_x = float(self.size_x_var.get())
        except Exception: size_x = 100.0
        try: nozzle = float(self.nozzle_var.get())
        except Exception: nozzle = 0.4
        try: layer  = float(self.layer_var.get())
        except Exception: layer = 0.12
        try: relief = float(self.relief_var.get())
        except Exception: relief = 3.0

        presets = calculate_presets(size_x, nozzle, layer, self._active_aspect_ratio())
        for name, data in presets.items():
            self.preset_descriptions[name].set(data["description"])

        chosen = self.selected_preset.get()
        p = presets[chosen]
        self.recommendation_var.set(summarize_image_recommendations(self.image_paths))
        if hasattr(self, "per_image_quality_frame"):
            self._refresh_per_image_quality_table()
        if not self.advanced_open.get():
            self.adv_vars["resolution"].set(str(p["resolution"]))
            self.adv_vars["blur"].set(str(p["blur"]))

    def _toggle_advanced(self):
        if self.advanced_open.get():
            self.adv_panel.grid(row=self.adv_panel_row, column=0, columnspan=3,
                                padx=14, pady=(0,4), sticky="ew")
        else:
            self.adv_panel.grid_remove()
            self._update_preset_display()

    #  File helpers 

    def _add_images(self):
        files = filedialog.askopenfilenames(
            title="Select images",
            filetypes=[("Image files", "*.png *.jpg *.jpeg *.tif *.tiff *.bmp"),
                       ("All files", "*.*")])
        for f in files:
            if f not in self.image_paths:
                self.image_paths.append(f)
                self.listbox.insert(tk.END, Path(f).name)
        self._update_preset_display()

    def _remove_selected(self):
        for i in reversed(list(self.listbox.curselection())):
            self.listbox.delete(i)
            self.image_paths.pop(i)
        self._update_preset_display()

    def _browse_output(self):
        folder = filedialog.askdirectory()
        if folder: self.output_var.set(folder)


    def _active_aspect_ratio(self):
        if self.image_paths:
            try:
                img = Image.open(self.image_paths[0]); w, h = img.size
                return max(0.1, h / w)
            except Exception:
                pass
        try:
            sx = float(self.size_x_var.get())
            sy_text = self.size_y_var.get().strip().lower()
            if sy_text not in ("", "auto"):
                return max(0.1, float(sy_text) / sx)
        except Exception:
            pass
        return 1.0

    def _get_params(self, img_path):
        try: size_x = float(self.size_x_var.get())
        except Exception: size_x = 100.0
        try: size_y = float(self.size_y_var.get())
        except Exception: size_y = smart_size_y(img_path, size_x)
        try: relief = float(self.relief_var.get())
        except Exception: relief = 3.0
        try: nozzle = float(self.nozzle_var.get())
        except Exception: nozzle = 0.4
        try: layer  = float(self.layer_var.get())
        except Exception: layer = 0.12

        presets = calculate_presets(size_x, nozzle, layer, self._active_aspect_ratio())
        preset_name = self.selected_preset.get()
        if self.per_image_quality_open.get() and img_path in self.per_image_quality_vars:
            preset_name = self.per_image_quality_vars[img_path].get()
        p = presets[preset_name]

        if self.advanced_open.get():
            try: resolution = int(self.adv_vars["resolution"].get())
            except Exception: resolution = p["resolution"]
            try: blur = float(self.adv_vars["blur"].get())
            except Exception: blur = p["blur"]
            try: base = float(self.adv_vars["base_thickness"].get())
            except Exception: base = BEST_PRACTICE["base_thickness"]
        else:
            resolution = p["resolution"]
            blur       = p["blur"]
            base       = BEST_PRACTICE["base_thickness"]

        return {"size_x": size_x, "size_y": size_y, "relief_height": relief,
                "base_thickness": base, "blur": blur, "resolution": resolution,
                "invert": self.invert_var.get(), "flip": self.flip_var.get()}

    def _get_formats(self):
        fmt = {"stl": self.fmt_stl.get(), "3mf": self.fmt_3mf.get(),
               "glb": self.fmt_glb.get()}
        if not any(fmt.values()):
            messagebox.showwarning("No format", "Select at least one output format.")
            return None
        return fmt

    #  Pipeline 

    def _run(self):
        if not self.image_paths:
            messagebox.showwarning("No images", "Add at least one image.")
            return
        formats = self._get_formats()
        if formats is None or self.running: return
        self.running = True
        self.run_btn.config(state=tk.DISABLED)
        threading.Thread(target=self._process_all, args=(formats,),
                         daemon=True).start()

    def _process_all(self, formats):
        output_dir = Path(self.output_var.get())
        output_dir.mkdir(parents=True, exist_ok=True)
        total = len(self.image_paths)
        errors = []
        for i, img_path in enumerate(self.image_paths):
            name = Path(img_path).stem
            self._set_status(f"Processing {i+1}/{total}: {name}...")
            self.progress["value"] = (i / total) * 100
            self.root.update_idletasks()
            try:
                params = self._get_params(img_path)
                hm = image_to_heightmap(
                    img_path,
                    output_size=(params["resolution"], params["resolution"]),
                    invert=params["invert"], blur_sigma=params["blur"],
                    preserve_aspect=True, flip=params["flip"])
                stl_path = output_dir / f"{name}_lithograph.stl"
                heightmap_to_stl(hm, stl_path,
                    base_thickness_mm=params["base_thickness"],
                    max_relief_mm=params["relief_height"],
                    physical_size_mm=(params["size_x"], params["size_y"]))
                convert_stl(stl_path, formats)
            except Exception as e:
                errors.append(f"{name}: {e}")
        self.progress["value"] = 100
        fmt_list = ", ".join(f".{k.upper()}" for k, v in formats.items() if v)
        if errors:
            self._set_status(f"Done with {len(errors)} error(s).")
            messagebox.showerror("Errors", "Some files failed:\n\n" + "\n".join(errors))
        else:
            self._set_status(f"Done! {total} file(s) as {fmt_list}")
            messagebox.showinfo("Done!",
                f"Generated {total} file(s) as {fmt_list}.\n\nSaved to:\n{output_dir}")
        self.running = False
        self.run_btn.config(state=tk.NORMAL)

    def _set_status(self, msg):
        self.status_var.set(msg)
        self.root.update_idletasks()


# Launch
root = tk.Tk()
app = TAMPBatchGUIv2(root)
root.mainloop()



  [i] 3717x3712 -> 128x128
  [OK] STL -> C:\Users\vazquez\TAMP_output\Asian_elephant_trunk_HeidelbergZoo_1_lithograph.stl
  [i] 3717x3712 -> 256x256
  [OK] STL -> C:\Users\vazquez\TAMP_output\Asian_elephant_trunk_HeidelbergZoo_1_lithograph.stl
  [i] 3717x3712 -> 512x511
  [OK] STL -> C:\Users\vazquez\TAMP_output\Asian_elephant_trunk_HeidelbergZoo_1_lithograph.stl
